In [ ]:
import os
import subprocess
import sys
import glob
import re
import shutil

# NLGCL Kaggle Runner v27 (Similarity CL - R1: pure-edge + boosted cl_reg)

# Helper to suppress output
def run_silent(cmd):
    if isinstance(cmd, list):
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    else:
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 1. Clone (Silent)
if not os.path.exists('main.py'):
    run_silent('git clone https://github.com/yangzeha/NLGCL.git')
    if os.path.exists('NLGCL'):
        os.chdir('NLGCL')

# 2. Patch Code (Silent) 
files = glob.glob('recbole/**/*.py', recursive=True) + glob.glob('recbole_gnn/**/*.py', recursive=True)

replacements = [
    (r'np\.float\b', 'float'),
    (r'np\.int\b', 'int'),
    (r'np\.object\b', 'object'),
    (r'np\.bool\b', 'bool'),
    (r'np\.str\b', 'str'),
    (r'np\.long\b', 'int'),
    # Fix RecBole Config does not support .get() — use bracket notation instead
    (r"config\.get\('pos_sim_threshold',\s*[0-9.]+\)",
     "config['pos_sim_threshold'] if 'pos_sim_threshold' in config else 0.1"),
    # Fix duplicate logging: clear existing root handlers before basicConfig
    (
        r'logging\.basicConfig\(level=level, handlers=\[sh, fh\]\)',
        'root_logger = logging.getLogger()\n    for _h in root_logger.handlers[:]:\n        root_logger.removeHandler(_h)\n        _h.close()\n    logging.basicConfig(level=level, handlers=[sh, fh])'
    ),
]

for file_path in files:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        new_content = content
        
        if 'def compatibility_settings(self):' in new_content:
            new_content = re.sub(
                r'def compatibility_settings\(self\):.*?(?=\n\s+def|\Z)', 
                'def compatibility_settings(self):\n        pass\n\n    ', 
                new_content, 
                flags=re.DOTALL
            )

        for pattern, replacement in replacements:
            new_content = re.sub(pattern, replacement, new_content)
            
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
    except Exception:
        pass

# 3. Setup Dependencies (Silent execution using subprocess)
setup_cmds = [
    [sys.executable, "-m", "pip", "install", "-q", "numpy>=2.0.0", "pandas>=2.2.2", "--force-reinstall", "--upgrade"],
    [sys.executable, "-m", "pip", "install", "-q", "torch==2.4.0", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu121"],
    [sys.executable, "-m", "pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", "https://data.pyg.org/whl/torch-2.4.0+cu121.html"]
]
for cmd in setup_cmds:
    run_silent(cmd)

requirements_content = """colorlog
tensorboard
six
colorama
pyyaml
hyperopt>=0.2.7
torch_geometric
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)
run_silent([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# 4. Dataset Preprocessing
preprocess_script = r"""
import os
import pandas as pd
import re
import shutil

DATASET_DIR = 'dataset/QB-video'
INTER_FILE = os.path.join(DATASET_DIR, 'QB-video.inter')
RAW_CSV_PATH = 'dataset/QB-video.csv'
FIELD_SEPARATOR = '\t'

os.makedirs(DATASET_DIR, exist_ok=True)

try:
    df = pd.read_csv(RAW_CSV_PATH, low_memory=False)
except:
    df = pd.read_csv(RAW_CSV_PATH, sep='\t', low_memory=False)

if 'click' in df.columns:
    df = df[df['click'] == 1].copy()

if 'video_id' in df.columns:
    df.rename(columns={'video_id': 'item_id'}, inplace=True)

df = df[['user_id', 'item_id']]

def strict_iterative_5core(data, k=5):
    while True:
        prev_len = len(data)
        item_counts = data.groupby('item_id')['user_id'].count()
        data = data[data['item_id'].isin(item_counts[item_counts >= k].index)]
        user_counts = data.groupby('user_id')['item_id'].count()
        data = data[data['user_id'].isin(user_counts[user_counts >= k].index)]
        if len(data) == prev_len:
            break
    return data

final_df = strict_iterative_5core(df, k=5)

u_count = final_df['user_id'].nunique()
i_count = final_df['item_id'].nunique()
inter_count = len(final_df)

print(f"[Final Match Result]")
print(f"Users: {u_count} (Expected ~30,324)")
print(f"Items: {i_count} (Expected ~25,731)")
print(f"Interactions: {inter_count} (Expected ~1,581,136)")

final_df.columns = ['user_id:token', 'item_id:token']
final_df.to_csv(INTER_FILE, sep='\t', index=False)
print(f"Saved processed data to {INTER_FILE}")

config_path = 'properties/QB-video.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        content = f.read()

    if 'user_inter_num_interval' in content:
        content = re.sub(r'user_inter_num_interval:.*', 'user_inter_num_interval: "[0,inf)"', content)
    else:
        content += '\nuser_inter_num_interval: "[0,inf)"'

    if 'item_inter_num_interval' in content:
        content = re.sub(r'item_inter_num_interval:.*', 'item_inter_num_interval: "[0,inf)"', content)
    else:
        content += '\nitem_inter_num_interval: "[0,inf)"'

    if 'field_separator' in content:
        content = re.sub(r'field_separator:.*', 'field_separator: "\\t"', content)
    else:
        content += '\nfield_separator: "\\t"'

    content = re.sub(r'val_interval:.*', '', content)

    # ----------------------------------------------------------------
    # Similarity-aware CL search — R1: pure-edge positive (threshold=0),
    # cl_reg boosted 10x vs best baseline, warm_up_step=50.
    # Baseline (no sim):  n_layers=3, cl_temp=0.1, cl_reg=1e-5 -> ndcg@10=0.0988
    # Search plan:
    #   R1  pos_sim_threshold=0.0,  cl_reg=1e-4,  warm_up_step=50
    #   R2  pos_sim_threshold=0.05, cl_reg=1e-4,  warm_up_step=50
    #   R3  pos_sim_threshold=0.1,  cl_reg=1e-4,  warm_up_step=100
    #   R4  pos_sim_threshold=0.0,  cl_reg=5e-4,  warm_up_step=50
    #   R5  pos_sim_threshold=0.1,  cl_reg=5e-4,  warm_up_step=100
    # ----------------------------------------------------------------
    params_to_inject = [
        "n_layers: 3",
        "embedding_size: 64",
        "reg_weight: 0.0001",
        "cl_temp: 0.1",
        "alpha: 0.6",
        "cl_reg: 0.0001",           # R1: 10x vs best baseline (binary CE weaker than InfoNCE)
        "pos_sim_threshold: 0.0",   # R1: pure edge-based positive, no sim filter cold-start issue
        "warm_up_step: 50",         # R1: longer warmup so embeddings develop before threshold
        "show_progress: False",
        "epochs: 500",
        "stopping_step: 30",
    ]
    for param in params_to_inject:
        key = param.split(':')[0]
        if key not in content:
            content += f"\n{param}"
        else:
            content = re.sub(f'{key}:.*', param, content)

    with open(config_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("Updated properties/QB-video.yaml.")
"""

with open('preprocess_dataset.py', 'w', encoding='utf-8') as f:
    f.write(preprocess_script)

subprocess.check_call([sys.executable, 'preprocess_dataset.py'])

# 5. Clear Cache and Start Training
if os.path.exists('saved'):
    shutil.rmtree('saved')
if os.path.exists('dataset_cache'):
    shutil.rmtree('dataset_cache')

print("Starting Training (R1: pos_sim_threshold=0.0, cl_reg=1e-4, warm_up_step=50)...")
subprocess.check_call([sys.executable, 'main.py', '--dataset', 'QB-video'])